In [1]:
# ============================================================
# End-to-End Hospital Operations Dashboard and SQL Analysis
# hospital_analysis.ipynb
# ============================================================


# ============================================================
# 1. Import Libraries
# ============================================================

import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt


# ============================================================
# 2. Load Dataset
# ============================================================

df = pd.read_csv("data/hospital_operations_dataset.csv")

df.head()


# ============================================================
# 3. Initial Data Exploration
# ============================================================

print("Dataset Shape:", df.shape)

df.info()

df.describe()

df.isnull().sum()

df.duplicated().sum()


# ============================================================
# 4. Data Cleaning
# ============================================================

# Fill missing Gender values with the most common gender
df["Gender"] = df["Gender"].fillna(df["Gender"].mode()[0])

# Fill missing Department values with "Unknown"
df["Department"] = df["Department"].fillna("Unknown")

# Fill missing Length of Stay with median
df["Length_of_Stay"] = df["Length_of_Stay"].fillna(df["Length_of_Stay"].median())

# Fill missing Oxygen Units Used with 0
df["Oxygen_Units_Used"] = df["Oxygen_Units_Used"].fillna(0)

# Remove duplicate rows
df = df.drop_duplicates()

# Convert Visit_Date to datetime
df["Visit_Date"] = pd.to_datetime(df["Visit_Date"])

# Create additional date columns
df["Year"] = df["Visit_Date"].dt.year
df["Month_Name"] = df["Visit_Date"].dt.month_name()
df["Weekday"] = df["Visit_Date"].dt.day_name()

# Create bed occupancy rate
df["Bed_Occupancy_Rate"] = (
    df["Beds_Occupied"] / df["Total_Beds_Available"]
) * 100

# Create ICU occupancy rate
df["ICU_Occupancy_Rate"] = (
    df["ICU_Beds_Occupied"] / df["ICU_Beds_Available"]
) * 100

# Convert readmission column into numeric flag
df["Readmission_Flag"] = df["Readmission_Within_30_Days"].map({
    "Yes": 1,
    "No": 0
})

# Final missing value check
df.isnull().sum()


# ============================================================
# 5. Cleaned Data Preview
# ============================================================

df.head()

df.columns


# ============================================================
# 6. Exploratory Data Analysis
# ============================================================


# Patient volume by department
department_volume = (
    df.groupby("Department")
    .size()
    .sort_values(ascending=False)
)

department_volume


# Average length of stay by department
avg_los_department = (
    df.groupby("Department")["Length_of_Stay"]
    .mean()
    .sort_values(ascending=False)
)

avg_los_department


# Readmission rate by department
readmission_rate_department = (
    df.groupby("Department")["Readmission_Flag"]
    .mean()
    .sort_values(ascending=False) * 100
)

readmission_rate_department


# Average bed occupancy by department
bed_occupancy_department = (
    df.groupby("Department")["Bed_Occupancy_Rate"]
    .mean()
    .sort_values(ascending=False)
)

bed_occupancy_department


# Average ICU occupancy by department
icu_occupancy_department = (
    df.groupby("Department")["ICU_Occupancy_Rate"]
    .mean()
    .sort_values(ascending=False)
)

icu_occupancy_department


# Top diseases
top_diseases = df["Disease"].value_counts().head(10)

top_diseases


# Severity distribution
severity_distribution = df["Severity"].value_counts()

severity_distribution


# Admission type distribution
admission_distribution = df["Admission_Type"].value_counts()

admission_distribution


# ============================================================
# 7. Basic Visualizations
# ============================================================

department_volume.plot(kind="bar", figsize=(10, 5), title="Patient Volume by Department")
plt.xlabel("Department")
plt.ylabel("Number of Patients")
plt.xticks(rotation=45)
plt.show()


avg_los_department.plot(kind="bar", figsize=(10, 5), title="Average Length of Stay by Department")
plt.xlabel("Department")
plt.ylabel("Average Length of Stay")
plt.xticks(rotation=45)
plt.show()


readmission_rate_department.plot(kind="bar", figsize=(10, 5), title="Readmission Rate by Department")
plt.xlabel("Department")
plt.ylabel("Readmission Rate (%)")
plt.xticks(rotation=45)
plt.show()


top_diseases.plot(kind="bar", figsize=(10, 5), title="Top 10 Diseases")
plt.xlabel("Disease")
plt.ylabel("Number of Patients")
plt.xticks(rotation=45)
plt.show()


severity_distribution.plot(kind="bar", figsize=(8, 5), title="Severity Distribution")
plt.xlabel("Severity")
plt.ylabel("Number of Patients")
plt.xticks(rotation=0)
plt.show()


# ============================================================
# 8. Create SQLite Database
# ============================================================

conn = sqlite3.connect("data/hospital_operations.db")

df.to_sql(
    "hospital_operations",
    conn,
    if_exists="replace",
    index=False
)

conn.close()

print("SQLite database created successfully.")


# ============================================================
# 9. Connect to SQLite Database
# ============================================================

conn = sqlite3.connect("data/hospital_operations.db")


# ============================================================
# 10. SQL Analysis
# ============================================================


# Total patient volume by department
query = """
SELECT
    Department,
    COUNT(*) AS Total_Patients
FROM hospital_operations
GROUP BY Department
ORDER BY Total_Patients DESC;
"""

pd.read_sql_query(query, conn)


# Average length of stay by department
query = """
SELECT
    Department,
    ROUND(AVG(Length_of_Stay), 2) AS Avg_Length_of_Stay
FROM hospital_operations
GROUP BY Department
ORDER BY Avg_Length_of_Stay DESC;
"""

pd.read_sql_query(query, conn)


# Readmission rate by department
query = """
SELECT
    Department,
    ROUND(AVG(Readmission_Flag) * 100, 2) AS Readmission_Rate
FROM hospital_operations
GROUP BY Department
ORDER BY Readmission_Rate DESC;
"""

pd.read_sql_query(query, conn)


# Average bed occupancy by department
query = """
SELECT
    Department,
    ROUND(AVG(Bed_Occupancy_Rate), 2) AS Avg_Bed_Occupancy_Rate
FROM hospital_operations
GROUP BY Department
ORDER BY Avg_Bed_Occupancy_Rate DESC;
"""

pd.read_sql_query(query, conn)


# Average ICU occupancy by department
query = """
SELECT
    Department,
    ROUND(AVG(ICU_Occupancy_Rate), 2) AS Avg_ICU_Occupancy_Rate
FROM hospital_operations
GROUP BY Department
ORDER BY Avg_ICU_Occupancy_Rate DESC;
"""

pd.read_sql_query(query, conn)


# Top diseases by patient count
query = """
SELECT
    Disease,
    COUNT(*) AS Total_Cases
FROM hospital_operations
GROUP BY Disease
ORDER BY Total_Cases DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)


# Average length of stay by disease
query = """
SELECT
    Disease,
    ROUND(AVG(Length_of_Stay), 2) AS Avg_Length_of_Stay
FROM hospital_operations
GROUP BY Disease
ORDER BY Avg_Length_of_Stay DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)


# Readmission rate by disease
query = """
SELECT
    Disease,
    ROUND(AVG(Readmission_Flag) * 100, 2) AS Readmission_Rate
FROM hospital_operations
GROUP BY Disease
ORDER BY Readmission_Rate DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)


# Patient volume by admission type
query = """
SELECT
    Admission_Type,
    COUNT(*) AS Total_Patients
FROM hospital_operations
GROUP BY Admission_Type
ORDER BY Total_Patients DESC;
"""

pd.read_sql_query(query, conn)


# Resource usage by severity
query = """
SELECT
    Severity,
    COUNT(*) AS Total_Patients,
    ROUND(AVG(Length_of_Stay), 2) AS Avg_Length_of_Stay,
    ROUND(AVG(Oxygen_Units_Used), 2) AS Avg_Oxygen_Units_Used,
    ROUND(AVG(Readmission_Flag) * 100, 2) AS Readmission_Rate
FROM hospital_operations
GROUP BY Severity
ORDER BY Total_Patients DESC;
"""

pd.read_sql_query(query, conn)


# Monthly patient volume
query = """
SELECT
    Year,
    Month,
    COUNT(*) AS Total_Patients
FROM hospital_operations
GROUP BY Year, Month
ORDER BY Year, Month;
"""

pd.read_sql_query(query, conn)


# Day of week patient volume
query = """
SELECT
    Day_of_Week,
    COUNT(*) AS Total_Patients
FROM hospital_operations
GROUP BY Day_of_Week
ORDER BY Total_Patients DESC;
"""

pd.read_sql_query(query, conn)


# Departments with high readmission and long stay
query = """
SELECT
    Department,
    COUNT(*) AS Total_Patients,
    ROUND(AVG(Length_of_Stay), 2) AS Avg_Length_of_Stay,
    ROUND(AVG(Readmission_Flag) * 100, 2) AS Readmission_Rate
FROM hospital_operations
GROUP BY Department
HAVING Total_Patients > 50
ORDER BY Readmission_Rate DESC, Avg_Length_of_Stay DESC;
"""

pd.read_sql_query(query, conn)


# ============================================================
# 11. Close Database Connection
# ============================================================

conn.close()

print("Analysis complete.")

FileNotFoundError: [Errno 2] No such file or directory: 'data/hospital_operations_dataset.csv'